# Notebook 3: Visualize What the Model Learned

This is where understanding gets concrete. We'll look at:
1. **Feature maps** — what each layer "sees" in a flower image
2. **Filters** — the actual learned patterns in early conv layers
3. **Grad-CAM** — where the model looks to make its decision

This proves that transfer learning works because early layers capture
universal visual patterns that apply beyond the original training domain.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from src.data import get_dataloaders, IMAGENET_MEAN, IMAGENET_STD
from src.model import create_model

In [ ]:
DEVICE = (
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)

# Load the fine-tuned model
model = create_model(num_classes=102, mode='fine_tune', unfreeze_from='layer3')
model.load_state_dict(torch.load('../models/fine_tune_best.pth', map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print('Model loaded.')

In [ ]:
# Get a batch of test images
_, _, test_loader = get_dataloaders(data_dir='../data', batch_size=8)
images, labels = next(iter(test_loader))

def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

# Show original images
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for i in range(8):
    axes[i].imshow(denormalize(images[i]).permute(1, 2, 0))
    axes[i].set_title(f'Label: {labels[i].item()}')
    axes[i].axis('off')
plt.suptitle('Input Images', fontsize=14)
plt.tight_layout()
plt.show()

## 1. Visualize Conv1 Filters

The first convolutional layer's filters are directly interpretable —
they show the basic patterns the network detects: edges, color blobs, gradients.

In [ ]:
# Extract first conv layer filters
filters = model.conv1.weight.data.cpu()
# Normalize each filter to [0, 1] for display
filters = (filters - filters.min()) / (filters.max() - filters.min())

fig, axes = plt.subplots(4, 16, figsize=(20, 5))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        # Show as RGB — these are 7x7x3 filters
        ax.imshow(filters[i].permute(1, 2, 0))
    ax.axis('off')

plt.suptitle('Conv1 Filters — Edge & Color Detectors (universal, not flower-specific)',
             fontsize=12)
plt.tight_layout()
plt.show()

## 2. Feature Maps at Different Depths

Watch how representations evolve from pixel-level to semantic:
- **Layer 1**: edges, simple textures
- **Layer 2**: more complex patterns
- **Layer 3-4**: high-level features (petal shapes, flower structure)

In [ ]:
# Hook into intermediate layers to capture feature maps
feature_maps = {}

def hook_fn(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

# Register hooks
hooks = []
for name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(model, name)
    hooks.append(layer.register_forward_hook(hook_fn(name)))

# Forward pass on one image
with torch.no_grad():
    sample = images[0:1].to(DEVICE)
    _ = model(sample)

# Remove hooks
for h in hooks:
    h.remove()

# Plot feature maps from each layer
fig, axes = plt.subplots(4, 8, figsize=(20, 10))
for row, name in enumerate(['layer1', 'layer2', 'layer3', 'layer4']):
    fmaps = feature_maps[name][0]  # First (only) image in batch
    for col in range(8):
        axes[row][col].imshow(fmaps[col], cmap='viridis')
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(name, fontsize=12, rotation=0, labelpad=50)

plt.suptitle('Feature Maps at Different Depths (8 channels each)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/feature_maps.png', dpi=150)
plt.show()

## 3. Grad-CAM — Where Does the Model Look?

Gradient-weighted Class Activation Mapping shows which image regions
were most important for the model's prediction. For flowers, it should
highlight petals and flower centers, NOT the background.

In [ ]:
class GradCAM:
    """Simple Grad-CAM implementation for learning purposes."""

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        # Hook to capture activations and gradients
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, image, class_idx=None):
        self.model.eval()
        output = self.model(image)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        # Backprop from the target class score
        self.model.zero_grad()
        score = output[0, class_idx]
        score.backward()

        # Weight each activation channel by its gradient (global avg pool)
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # GAP
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)  # Only positive contributions
        cam = F.interpolate(cam, size=image.shape[2:], mode='bilinear',
                           align_corners=False)

        # Normalize to [0, 1]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam[0, 0].cpu().numpy(), class_idx

In [ ]:
# Apply Grad-CAM to several test images
grad_cam = GradCAM(model, model.layer4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = images[i:i+1].to(DEVICE).requires_grad_(True)
    cam, pred_class = grad_cam(img)

    # Original image
    axes[0][i].imshow(denormalize(images[i]).permute(1, 2, 0))
    axes[0][i].set_title(f'True: {labels[i].item()}')
    axes[0][i].axis('off')

    # Grad-CAM overlay
    axes[1][i].imshow(denormalize(images[i]).permute(1, 2, 0))
    axes[1][i].imshow(cam, cmap='jet', alpha=0.5)
    axes[1][i].set_title(f'Pred: {pred_class}')
    axes[1][i].axis('off')

axes[0][0].set_ylabel('Original', fontsize=12)
axes[1][0].set_ylabel('Grad-CAM', fontsize=12)
plt.suptitle('Where does the model look? (red = high attention)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/gradcam.png', dpi=150)
plt.show()

## What You Should Observe

1. **Conv1 filters** look like generic edge/color detectors — identical
   whether trained on ImageNet or fine-tuned on flowers. These are universal.

2. **Feature maps get more abstract** deeper in the network — from
   edge-like patterns (layer1) to flower-part-like activations (layer4).

3. **Grad-CAM highlights the flower**, not the background — the model
   learned what matters even though it was originally trained on ImageNet.

This is the **core insight of transfer learning**: the features learned on
one large dataset are reusable across different visual domains.